# Create BioASQ retrieval subsets

This notebook combines the expert-authored BioASQ 11b questions and gold PubMed document annotations with a corpus of PubMed titles and abstracts. It can create separate retrieval samples by question type and by whether a question has one or multiple relevant documents.

Only questions whose complete gold-document set is present in the corpus are eligible. This avoids silently converting a multi-document question into a single-document question.

In [1]:
import sys
from pathlib import Path

REPO_ROOT = Path("/content/retrieval-benchlab")
if "google.colab" in sys.modules:
    if not (REPO_ROOT / ".git").exists():
        !git clone --depth 1 https://github.com/lohex/retrieval-benchlab.git {REPO_ROOT}
    %cd {REPO_ROOT}
else:
    REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
    if str(REPO_ROOT) not in sys.path:
        sys.path.insert(0, str(REPO_ROOT))

!pip -q install -U datasets tqdm


Cloning into '/content/retrieval-benchlab'...
remote: Enumerating objects: 29, done.
remote: Counting objects: 100% (29/29), done.
remote: Compressing objects: 100% (29/29), done.
remote: Total 29 (delta 0), reused 10 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (29/29), 47.06 KiB | 3.14 MiB/s, done.
/content/retrieval-benchlab
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.3/57.3 kB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 559.1/559.1 kB 18.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 80.2/80.2 kB 6.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/50.1 MB 18.7 MB/s eta 0:00:00


In [2]:
import logging

from src.dataset_builder import (
    create_bioasq_calibration_set,
    create_bioasq_sample,
)
from src.io import load_bioasq_benchmark, mount_google_drive

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(message)s",
    force=True,
)
logger = logging.getLogger("bioasq-sample")


## Configuration

`N_QUERIES_PER_SUBSET = 90` keeps all six subsets equally sized. Set it to `None` to use every eligible question in each subset.

In [3]:
CORPUS_DATASET_NAME = "DinoStackAI/bioasq-rag-13b-resplit"
CORPUS_CONFIG = "corpus"
CORPUS_SPLIT = "train"
QUESTIONS_SOURCE = "https://zenodo.org/api/records/7655130/files/training11b.json/content"

N_QUERIES_PER_SUBSET = 90
N_CORPUS_DOCS = 30_000
N_CALIBRATION_DOCS = 5_000
SEED = 42
CALIBRATION_SEED = 43

OUTPUT_ROOT = "/content/drive/MyDrive/Retreaval/data"
CALIBRATION_OUTPUT_DIR = (
    "/content/drive/MyDrive/Retreaval/calibration/bioasq-5k"
)
CALIBRATION_QUESTION_TYPES = ("list", "factoid", "summary")


## Load the shared benchmark once

The source corpus is loaded only once and reused by the calibration and subset-creation blocks. Empty source documents are discarded while loading.

In [4]:
mount_google_drive()
benchmark = load_bioasq_benchmark(
    corpus_dataset_name=CORPUS_DATASET_NAME,
    corpus_config=CORPUS_CONFIG,
    corpus_split=CORPUS_SPLIT,
    questions_source=QUESTIONS_SOURCE,
)


2026-09-06 21:02:29,865 | INFO | Mounting Google Drive
2026-09-06 21:02:48,185 | INFO | Loading corpus DinoStackAI/bioasq-rag-13b-resplit/corpus[train]


Mounted at /content/drive


2026-09-06 21:02:48,794 | INFO | HTTP Request: GET https://huggingface.co/api/agent-harnesses "HTTP/1.1 200 OK"
2026-09-06 21:02:48,885 | INFO | HTTP Request: HEAD https://huggingface.co/datasets/DinoStackAI/bioasq-rag-13b-resplit/resolve/main/README.md "HTTP/1.1 307 Temporary Redirect"
2026-09-06 21:02:48,887 | WARNING | Warning: You are sending unauthenticated requests to the HF Hub. Please set a HF_TOKEN to enable higher rate limits and faster downloads.
2026-09-06 21:02:48,974 | INFO | HTTP Request: HEAD https://huggingface.co/api/resolve-cache/datasets/DinoStackAI/bioasq-rag-13b-resplit/ade805f1df0b16dbf19385597861146bdb0d9904/README.md "HTTP/1.1 200 OK"
2026-09-06 21:02:49,077 | INFO | HTTP Request: GET https://huggingface.co/api/resolve-cache/datasets/DinoStackAI/bioasq-rag-13b-resplit/ade805f1df0b16dbf19385597861146bdb0d9904/README.md "HTTP/1.1 200 OK"


README.md:   0%|          | 0.00/3.69k [00:00<?, ?B/s]

2026-09-06 21:02:49,201 | INFO | HTTP Request: HEAD https://huggingface.co/datasets/DinoStackAI/bioasq-rag-13b-resplit/resolve/ade805f1df0b16dbf19385597861146bdb0d9904/bioasq-rag-13b-resplit.py "HTTP/1.1 404 Not Found"
2026-09-06 21:02:49,452 | INFO | HTTP Request: HEAD https://s3.amazonaws.com/datasets.huggingface.co/datasets/datasets/DinoStackAI/bioasq-rag-13b-resplit/DinoStackAI/bioasq-rag-13b-resplit.py "HTTP/1.1 404 Not Found"
2026-09-06 21:02:49,605 | INFO | HTTP Request: GET https://huggingface.co/api/datasets/DinoStackAI/bioasq-rag-13b-resplit/revision/ade805f1df0b16dbf19385597861146bdb0d9904 "HTTP/1.1 200 OK"
2026-09-06 21:02:49,695 | INFO | HTTP Request: HEAD https://huggingface.co/datasets/DinoStackAI/bioasq-rag-13b-resplit/resolve/ade805f1df0b16dbf19385597861146bdb0d9904/.huggingface.yaml "HTTP/1.1 404 Not Found"
2026-09-06 21:02:49,899 | INFO | HTTP Request: GET https://datasets-server.huggingface.co/info?dataset=DinoStackAI/bioasq-rag-13b-resplit "HTTP/1.1 200 OK"
2026-09

corpus/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 39.3MB            

corpus/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/44183 [00:00<?, ? examples/s]

Loading corpus:   0%|          | 0/44183 [00:00<?, ?doc/s]

2026-09-06 21:02:57,344 | INFO | Downloading BioASQ question metadata from https://zenodo.org/api/records/7655130/files/training11b.json/content
2026-09-06 21:03:01,419 | INFO | Loaded 4719 expert questions with type counts {'factoid': 1417, 'list': 901, 'summary': 1130, 'yesno': 1271}
2026-09-06 21:03:04,694 | WARNING | Excluding 1886 questions with at least one missing gold document
2026-09-06 21:03:04,700 | INFO | Benchmark ready: 2833 complete questions and 44128 corpus documents


## Create the shared calibration set

The calibration set contains 5,000 documents that are not relevant to any complete `list`, `factoid`, or `summary` question. It is stored outside the retrieval-dataset directory and excluded from every sampled evaluation corpus.

In [5]:
created_calibration = create_bioasq_calibration_set(
    benchmark,
    n_documents=N_CALIBRATION_DOCS,
    seed=CALIBRATION_SEED,
    output_dir=CALIBRATION_OUTPUT_DIR,
    protected_question_types=CALIBRATION_QUESTION_TYPES,
)
calibration_document_ids = frozenset(
    created_calibration.calibration_set.corpus
)
print(
    f"Created calibration set: {created_calibration.output_dir} "
    f"({len(calibration_document_ids)} documents)"
)


Building calibration set:   0%|          | 0/5000 [00:00<?, ?doc/s]

2026-09-06 21:03:04,771 | INFO | Built calibration set with 5000 documents, excluding 10610 documents relevant to question types ['factoid', 'list', 'summary']
2026-09-06 21:03:05,455 | INFO | Writing new document store to /content/drive/MyDrive/Retreaval/calibration/bioasq-5k_building


Saving the dataset (0/1 shards):   0%|          | 0/5000 [00:00<?, ? examples/s]

2026-09-06 21:03:05,802 | INFO | New document store saved successfully to /content/drive/MyDrive/Retreaval/calibration/bioasq-5k


Created calibration set: /content/drive/MyDrive/Retreaval/calibration/bioasq-5k (5000 documents)


## Shared builder arguments

The creation functions are imported from `src.dataset_builder`. This cell collects the arguments shared by all six retrieval subsets.

In [6]:
sample_creation_kwargs = {
    "benchmark": benchmark,
    "n_queries": N_QUERIES_PER_SUBSET,
    "n_corpus_docs": N_CORPUS_DOCS,
    "seed": SEED,
    "output_root": OUTPUT_ROOT,
    "calibration_document_ids": calibration_document_ids,
    "calibration_set_path": created_calibration.output_dir,
}


## Create `list` subsets

This block creates `list-one` and `list-multiple`.

In [7]:
created_subsets = globals().get("created_subsets", {})
for document_filter in ("one", "multiple"):
    key = f"list-{document_filter}"
    created_subsets[key] = create_bioasq_sample(
        question_type="list",
        documents=document_filter,
        subset_name=key,
        **sample_creation_kwargs,
    )
    print(
        f"Created {key}: {created_subsets[key].output_dir} "
        f"({created_subsets[key].dataset_id})"
    )


2026-09-06 21:03:13,212 | INFO | Selecting questions for type=list and documents=one
2026-09-06 21:03:13,219 | INFO | Selected 90 of 94 eligible questions for type=list, documents=one
2026-09-06 21:03:13,220 | INFO | Sampling random negative documents


Building corpus:   0%|          | 0/30000 [00:00<?, ?doc/s]

2026-09-06 21:03:13,310 | INFO | Built corpus with 89 positives, 29911 random negatives, and 5000 excluded documents
2026-09-06 21:03:13,329 | INFO | Validated sample: 90 queries, 89 positives, 30000 corpus documents
2026-09-06 21:03:13,337 | INFO | Writing new document store to /content/drive/MyDrive/Retreaval/data/list-one_building


Saving the dataset (0/1 shards):   0%|          | 0/30000 [00:00<?, ? examples/s]

2026-09-06 21:03:14,351 | INFO | New document store saved successfully to /content/drive/MyDrive/Retreaval/data/list-one
2026-09-06 21:03:14,352 | INFO | Loading sample from /content/drive/MyDrive/Retreaval/data/list-one
2026-09-06 21:03:15,379 | INFO | Loaded 90 queries, 89 positive documents and 30000 corpus documents
2026-09-06 21:03:16,290 | INFO | Dataset registered as dataset_fda0ded9db1077f6f3a900fe
2026-09-06 21:03:16,291 | INFO | Selecting questions for type=list and documents=multiple
2026-09-06 21:03:16,297 | INFO | Selected 90 of 372 eligible questions for type=list, documents=multiple
2026-09-06 21:03:16,299 | INFO | Sampling random negative documents


...

Building corpus:   0%|          | 0/30000 [00:00<?, ?doc/s]

2026-09-06 21:03:16,395 | INFO | Built corpus with 616 positives, 29384 random negatives, and 5000 excluded documents
2026-09-06 21:03:16,415 | INFO | Validated sample: 90 queries, 616 positives, 30000 corpus documents
2026-09-06 21:03:16,424 | INFO | Writing new document store to /content/drive/MyDrive/Retreaval/data/list-multiple_building


Saving the dataset (0/1 shards):   0%|          | 0/30000 [00:00<?, ? examples/s]

2026-09-06 21:03:17,411 | INFO | New document store saved successfully to /content/drive/MyDrive/Retreaval/data/list-multiple
2026-09-06 21:03:17,413 | INFO | Loading sample from /content/drive/MyDrive/Retreaval/data/list-multiple
2026-09-06 21:03:18,433 | INFO | Loaded 90 queries, 616 positive documents and 30000 corpus documents
2026-09-06 21:03:18,770 | INFO | Dataset registered as dataset_5a65dc9c69f670907636f769


Created list-multiple: /content/drive/MyDrive/Retreaval/data/list-multiple (dataset_5a65dc9c69f670907636f769)


## Create `factoid` subsets

This block creates `factoid-one` and `factoid-multiple`.

In [8]:
created_subsets = globals().get("created_subsets", {})
for document_filter in ("one", "multiple"):
    key = f"factoid-{document_filter}"
    created_subsets[key] = create_bioasq_sample(
        question_type="factoid",
        documents=document_filter,
        subset_name=key,
        **sample_creation_kwargs,
    )
    print(
        f"Created {key}: {created_subsets[key].output_dir} "
        f"({created_subsets[key].dataset_id})"
    )


2026-09-06 21:03:18,781 | INFO | Selecting questions for type=factoid and documents=one
2026-09-06 21:03:18,786 | INFO | Selected 90 of 337 eligible questions for type=factoid, documents=one
2026-09-06 21:03:18,787 | INFO | Sampling random negative documents


Building corpus:   0%|          | 0/30000 [00:00<?, ?doc/s]

2026-09-06 21:03:18,874 | INFO | Built corpus with 89 positives, 29911 random negatives, and 5000 excluded documents
2026-09-06 21:03:18,894 | INFO | Validated sample: 90 queries, 89 positives, 30000 corpus documents
2026-09-06 21:03:18,902 | INFO | Writing new document store to /content/drive/MyDrive/Retreaval/data/factoid-one_building


Saving the dataset (0/1 shards):   0%|          | 0/30000 [00:00<?, ? examples/s]

2026-09-06 21:03:20,481 | INFO | New document store saved successfully to /content/drive/MyDrive/Retreaval/data/factoid-one
2026-09-06 21:03:20,484 | INFO | Loading sample from /content/drive/MyDrive/Retreaval/data/factoid-one
2026-09-06 21:03:21,891 | INFO | Loaded 90 queries, 89 positive documents and 30000 corpus documents
2026-09-06 21:03:22,694 | INFO | Dataset registered as dataset_ca99cde454406cf2461ff087
2026-09-06 21:03:22,696 | INFO | Selecting questions for type=factoid and documents=multiple
2026-09-06 21:03:22,702 | INFO | Selected 90 of 559 eligible questions for type=factoid, documents=multiple
2026-09-06 21:03:22,703 | INFO | Sampling random negative documents


...

Building corpus:   0%|          | 0/30000 [00:00<?, ?doc/s]

2026-09-06 21:03:22,830 | INFO | Built corpus with 659 positives, 29341 random negatives, and 5000 excluded documents
2026-09-06 21:03:22,847 | INFO | Validated sample: 90 queries, 659 positives, 30000 corpus documents
2026-09-06 21:03:22,862 | INFO | Writing new document store to /content/drive/MyDrive/Retreaval/data/factoid-multiple_building


Saving the dataset (0/1 shards):   0%|          | 0/30000 [00:00<?, ? examples/s]

2026-09-06 21:03:24,043 | INFO | New document store saved successfully to /content/drive/MyDrive/Retreaval/data/factoid-multiple
2026-09-06 21:03:24,046 | INFO | Loading sample from /content/drive/MyDrive/Retreaval/data/factoid-multiple
2026-09-06 21:03:25,095 | INFO | Loaded 90 queries, 659 positive documents and 30000 corpus documents
2026-09-06 21:03:25,460 | INFO | Dataset registered as dataset_926e6a6c40da888d776df325


Created factoid-multiple: /content/drive/MyDrive/Retreaval/data/factoid-multiple (dataset_926e6a6c40da888d776df325)


## Create `summary` subsets

This block creates `summary-one` and `summary-multiple`.

In [9]:
created_subsets = globals().get("created_subsets", {})
for document_filter in ("one", "multiple"):
    key = f"summary-{document_filter}"
    created_subsets[key] = create_bioasq_sample(
        question_type="summary",
        documents=document_filter,
        subset_name=key,
        **sample_creation_kwargs,
    )
    print(
        f"Created {key}: {created_subsets[key].output_dir} "
        f"({created_subsets[key].dataset_id})"
    )


2026-09-06 21:03:25,476 | INFO | Selecting questions for type=summary and documents=one
2026-09-06 21:03:25,481 | INFO | Selected 90 of 252 eligible questions for type=summary, documents=one
2026-09-06 21:03:25,483 | INFO | Sampling random negative documents


Building corpus:   0%|          | 0/30000 [00:00<?, ?doc/s]

2026-09-06 21:03:25,564 | INFO | Built corpus with 88 positives, 29912 random negatives, and 5000 excluded documents
2026-09-06 21:03:25,583 | INFO | Validated sample: 90 queries, 88 positives, 30000 corpus documents
2026-09-06 21:03:25,599 | INFO | Writing new document store to /content/drive/MyDrive/Retreaval/data/summary-one_building


Saving the dataset (0/1 shards):   0%|          | 0/30000 [00:00<?, ? examples/s]

2026-09-06 21:03:26,735 | INFO | New document store saved successfully to /content/drive/MyDrive/Retreaval/data/summary-one
2026-09-06 21:03:26,737 | INFO | Loading sample from /content/drive/MyDrive/Retreaval/data/summary-one
2026-09-06 21:03:27,604 | INFO | Loaded 90 queries, 88 positive documents and 30000 corpus documents
2026-09-06 21:03:27,892 | INFO | Dataset registered as dataset_2be1ecbc4cc7a4b34af07af7
2026-09-06 21:03:27,894 | INFO | Selecting questions for type=summary and documents=multiple
2026-09-06 21:03:27,900 | INFO | Selected 90 of 465 eligible questions for type=summary, documents=multiple
2026-09-06 21:03:27,902 | INFO | Sampling random negative documents


...

Building corpus:   0%|          | 0/30000 [00:00<?, ?doc/s]

2026-09-06 21:03:27,993 | INFO | Built corpus with 621 positives, 29379 random negatives, and 5000 excluded documents
2026-09-06 21:03:28,012 | INFO | Validated sample: 90 queries, 621 positives, 30000 corpus documents
2026-09-06 21:03:28,020 | INFO | Writing new document store to /content/drive/MyDrive/Retreaval/data/summary-multiple_building


Saving the dataset (0/1 shards):   0%|          | 0/30000 [00:00<?, ? examples/s]

2026-09-06 21:03:29,142 | INFO | New document store saved successfully to /content/drive/MyDrive/Retreaval/data/summary-multiple
2026-09-06 21:03:29,143 | INFO | Loading sample from /content/drive/MyDrive/Retreaval/data/summary-multiple
2026-09-06 21:03:30,008 | INFO | Loaded 90 queries, 621 positive documents and 30000 corpus documents
2026-09-06 21:03:30,289 | INFO | Dataset registered as dataset_7d38fc53677402a92c5c06c4


Created summary-multiple: /content/drive/MyDrive/Retreaval/data/summary-multiple (dataset_7d38fc53677402a92c5c06c4)


## Browse subset examples

Change `EXAMPLE_SUBSET` and `EXAMPLE_PAGE`, then rerun the cell. The selected subset must have been created in this runtime.

In [10]:
from IPython.display import Markdown, display

EXAMPLE_SUBSET = "list-multiple"
EXAMPLE_PAGE = 0
EXAMPLES_PER_PAGE = 3

if EXAMPLE_SUBSET not in created_subsets:
    raise KeyError(f"Create {EXAMPLE_SUBSET!r} before browsing it")
created_sample = created_subsets[EXAMPLE_SUBSET]
sample = created_sample.sample
queries = sample.queries
relevant_docs = sample.relevant_docs
corpus = sample.corpus
metadata = sample.metadata
query_items = list(queries.items())
n_pages = max(1, (len(query_items) + EXAMPLES_PER_PAGE - 1) // EXAMPLES_PER_PAGE)
page = EXAMPLE_PAGE % n_pages
start = page * EXAMPLES_PER_PAGE

display(Markdown(
    f"### {EXAMPLE_SUBSET}: page {page + 1} of {n_pages}  "
    f"\nEligible questions: {metadata['n_eligible_queries']}"
))
for qid, query in query_items[start:start + EXAMPLES_PER_PAGE]:
    snippets = []
    for doc_id in sorted(relevant_docs[qid]):
        text = corpus[doc_id].replace("\n", " ")
        suffix = "…" if len(text) > 700 else ""
        snippets.append(f"**Document `{doc_id}`:** {text[:700]}{suffix}")
    display(Markdown(
        f"**Query `{qid}` ({metadata['query_types'][qid]}):** {query}\n\n"
        + "\n\n".join(snippets)
    ))


### list-multiple: page 1 of 30  
Eligible questions: 372

**Query `5e92021a2d3121100d000005` (list):** List the essential aminoacids.

**Document `11347201`:** [Nutritional status in adults on an alternative or traditional diet]. BACKGROUND: Plant food lacks vitamin B12, vitamin D and higher n-3 polyunsaturated fatty acids. Essential aminoacids methionine and lysine can be found in significantly lower amounts. On the contrary, the culinary and technologically non-processed plant food and whole-grain products contain essential nutrients in a highly condensed form. The aim of the study was to compare nutritional status of adults on alternative or on traditional diet and sequels of the diet to body metabolism. METHODS AND RESULTS: The group on alternative diet consisted of 89 lacto-ovo-vegetarians (age 38.7 +/- 0.6 years, average duration of vegetaria…

**Document `23477202`:** [Alpha-Lactalbumin as an ingredient of infant formula]. Alpha-Lactalbumin is the main whey protein in human milk rising 2,44 g/L in mature milk. It has a key function in the synthesis of lactose from glucose and galactose in the mammary gland although this compound has also other beneficial effects on the infant health due to the high proportion of essential aminoacids (tryptophan and cysteine). It seems also to increase iron absorption in the digestive track, and in in vitro experiments, linked to oleic acid (HAMLET complex), has shown anticarcinogenic effects against cellular tumor such as human papilloma. In addition, this complex has been reported to exhibit antimicrobial properties agai…

**Document `28089725`:** High concentration of branched-chain amino acids promotes oxidative stress, inflammation and migration of human peripheral blood mononuclear cells via mTORC1 activation. Leucine, isoleucine and valine are essential aminoacids termed branched-chain amino acids (BCAA) due to its aliphatic side-chain. In several pathological and physiological conditions increased BCAA plasma concentrations have been described. Elevated BCAA levels predict insulin resistance development. Moreover, BCAA levels higher than 2mmol/L are neurotoxic by inducing microglial activation in maple syrup urine disease. However, there are no studies about the direct effects of BCAA in circulating cells. We have explored wheth…

**Document `6620854`:** Oral essential aminoacid and ketoacid supplements in children with chronic renal failure. The effects on growth, body composition, and metabolism of a protein-restricted diet supplemented with essential aminoacids, the calcium-ketoacids of valine, leucine, isoleucine, and phenylalanine, and the calcium-hydroxyacid of methionine, were investigated in seven growth-retarded children with chronic renal failure. During 0.4 to 1.0 years of treatment there were significant increases in growth velocity and upper arm circumference SD scores, body cell mass (intracellular water calculated as tritium space minus corrected sodium bromide space) and serum transferrin. Blood urea and urea:creatinine ratio…

**Query `5324cca79b2d7acc7e00001d` (list):** What are the mobile applications fields of use for patients ?

**Document `21591562`:** mHealth. More than 17,000 mHealth apps now are available for smart phones and other devices, and they do everything from monitoring urine flow for patients with enlarged prostates to reminding people prone to kidney stones to drink more water. And that's just on the consumer side. Countless apps are avail-able for use by clinicians and hospitals. Mobile apps are changing health care in dramatic ways.

**Document `21689119`:** Promoting behavior change from alcohol use through mobile technology: the future of ecological momentary assessment. BACKGROUND: Interactive and mobile technologies (i.e., smartphones such as Blackberries, iPhones, and palm-top computers) show promise as an efficacious and cost-effective means of communicating health-behavior risks, improving public health outcomes, and accelerating behavior change. The present study was conducted as a "needs assessment" to examine the current available mobile smartphone applications (e.g., apps) that utilize principles of ecological momentary assessment (EMA)-daily self-monitoring or near real-time self-assessment of alcohol-use behavior-to promote positive…

**Document `22942063`:** Personalised mobile health and fitness apps: lessons learned from myFitnessCompanion®. Smartphones and tablets are slowly but steadily changing the way we look after our health and fitness. Today, many high quality mobile apps are available for users and health professionals and cover the whole health care chain, i.e. information collection, prevention, diagnosis, treatment and monitoring. Our team has developed a mobile health and fitness app called myFitnessCompanion® which has been available via Android market since February 2011. The objective of this paper is to share our experience with rolling out a mobile health and fitness app. We discuss the acceptance of health apps by end-users a…

**Document `23821609`:** Medical applications for pharmacists using mobile devices. BACKGROUND: Mobile devices (eg, smartphones, tablet computers) have become ubiquitous and subsequently there has been a growth in mobile applications (apps). Concurrently, mobile devices have been integrated into health care practice due to the availability and quality of medical apps. These mobile medical apps offer increased access to clinical references and point-of-care tools. However, there has been little identification of mobile medical apps suitable for the practice of pharmacy. OBJECTIVE: To address the shortage of recommendations of mobile medical apps for pharmacists in daily practice. DATA SOURCES: Mobile medical apps wer…

**Document `24067948`:** Mobile applications in dermatology. IMPORTANCE: With advancements in mobile technology, cellular phone-based mobile applications (apps) may be used in the practice and delivery of dermatologic care. OBJECTIVE: To identify and categorize the variety of current mobile apps available in dermatology for patients and providers. DESIGN, SETTING, AND PARTICIPANTS: Dermatology-related search terms were queried in the online app stores of the most commonly used mobile platforms developed by Apple, Android, Blackberry, Nokia, and Windows. Applications were assigned to categories based on description. Popularity, price, and reviews were recorded and target audiences were determined through websites off…

**Document `24073184`:** Mobile apps for pediatric obesity prevention and treatment, healthy eating, and physical activity promotion: just fun and games? Mobile applications (apps) offer a novel way to engage children in behavior change, but little is known about content of commercially available apps for this population. We analyzed the content of apps for iPhone/iPad for pediatric weight loss, healthy eating (HE), and physical activity (PA). Fifty-seven apps were downloaded and tested by two independent raters. Apps were coded for: inclusion of the Expert Committee for Pediatric Obesity Prevention's (ECPOP) eight recommended strategies (e.g., set goals) and seven behavioral targets (e.g., do ≥1 h of PA per day), u…

**Document `24139770`:** Evidence-based strategies in weight-loss mobile apps. BACKGROUND: Physicians have limited time for weight-loss counseling, and there is a lack of resources to which they can refer patients for assistance with weight loss. Weight-loss mobile applications (apps) have the potential to be a helpful tool, but the extent to which they include the behavioral strategies included in evidence-based interventions is unknown. PURPOSE: The primary aims of the study were to determine the degree to which commercial weight-loss mobile apps include the behavioral strategies included in evidence-based weight-loss interventions, and to identify features that enhance behavioral strategies via technology. METHOD…

**Query `5e4027f948dab47f2600000d` (list):** List diseases that are caused by the Meningococcus B?

**Document `12642606`:** Vaccination against Neisseria meningitidis using three variants of the lipoprotein GNA1870. Sepsis and meningitis caused by serogroup B meningococcus are devastating diseases of infants and young adults, which cannot yet be prevented by vaccination. By genome mining, we discovered GNA1870, a new surface-exposed lipoprotein of Neisseria meningitidis that induces high levels of bactericidal antibodies. The antigen is expressed by all strains of N. meningitidis tested. Sequencing of the gene in 71 strains representative of the genetic and geographic diversity of the N. meningitidis population, showed that the protein can be divided into three variants. Conservation within each variant ranges be…

**Document `16825336`:** A universal vaccine for serogroup B meningococcus. Meningitis and sepsis caused by serogroup B meningococcus are two severe diseases that still cause significant mortality. To date there is no universal vaccine that prevents these diseases. In this work, five antigens discovered by reverse vaccinology were expressed in a form suitable for large-scale manufacturing and formulated with adjuvants suitable for human use. The vaccine adjuvanted by aluminum hydroxide induced bactericidal antibodies in mice against 78% of a panel of 85 meningococcal strains representative of the global population diversity. The strain coverage could be increased to 90% and above by the addition of CpG oligonucleoti…

**Document `28778616`:** Emerging clinical experience with vaccines against group B meningococcal disease. The prevention of paediatric bacterial meningitis and septicaemia has recently entered a new era with the availability of two vaccines against capsular group B meningococcus (MenB). Both of these vaccines are based on sub-capsular proteins of the meningococcus, an approach that overcomes the challenges set by the poorly immunogenic MenB polysaccharide capsule but adds complexity to predicting and measuring the impact of their use. This review describes the development and use of MenB vaccines to date, from the use of outer membrane vesicle (OMV) vaccines in MenB outbreaks around the world, to emerging evidence …